# Task 03 – Feature Engineering & Preprocessing Pipeline

**Objective:** Build a robust, leak-free preprocessing workflow for a tabular credit-risk dataset using reusable Scikit-learn pipelines.

This notebook demonstrates:
- Loading a tabular dataset containing numerical and categorical columns
- Train/test splitting **before** fitting any preprocessing transformations
- Missing-value imputation, numerical scaling, and categorical one-hot encoding
- A `ColumnTransformer` combined with a model in a single `Pipeline`
- Feature-importance analysis using mutual information and a tree-based model

> Dataset: `credit_risk_sample.csv` (provided with this notebook). It is a reproducible sample dataset for demonstration and learning; it is not real customer or banking data.

## 1. Import required libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import mutual_info_classif

## 2. Load and inspect the dataset
Place `credit_risk_sample.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv("credit_risk_sample.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\\nMissing values:")
display(df.isnull().sum())
print("\\nTarget distribution:")
display(df["loan_default"].value_counts(normalize=True).rename("proportion"))

## 3. Separate features and target

In [ ]:
target = "loan_default"
X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

## 4. Split the data before preprocessing

The test set is held out before imputers, scalers, or encoders are fitted. This prevents information from the test set leaking into training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 5. Build preprocessing pipelines

- Numerical columns: median imputation followed by standard scaling.
- Categorical columns: most-frequent imputation followed by one-hot encoding.
- `handle_unknown="ignore"` allows the model to handle categories that appear in test/new data but were not present during training.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

## 6. Create one end-to-end machine learning pipeline
The preprocessing steps are fitted only on training data when `model_pipeline.fit()` is called.

In [ ]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=4,
        random_state=42,
        class_weight="balanced"
    ))
])

model_pipeline.fit(X_train, y_train)

## 7. Evaluate the model on unseen test data

In [ ]:
y_pred = model_pipeline.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

## 8. Feature importance – tree-based ranking

The fitted preprocessing pipeline provides the transformed feature names. Random Forest feature importances can then be mapped back to those names.

In [ ]:
feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = model_pipeline.named_steps["classifier"].feature_importances_

tree_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)
display(tree_importance.head(15).rename("random_forest_importance").to_frame())

tree_importance.head(15).sort_values().plot(kind="barh", figsize=(9, 6))
plt.title("Top 15 features – Random Forest importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 9. Feature ranking – mutual information

Mutual information measures the dependency between each transformed feature and the target. It is calculated on the training data only.

In [ ]:
X_train_transformed = model_pipeline.named_steps["preprocessor"].transform(X_train)
mi_scores = mutual_info_classif(
    X_train_transformed,
    y_train,
    random_state=42,
    discrete_features="auto"
)
mi_importance = pd.Series(mi_scores, index=feature_names).sort_values(ascending=False)
display(mi_importance.head(15).rename("mutual_information").to_frame())

mi_importance.head(15).sort_values().plot(kind="barh", figsize=(9, 6))
plt.title("Top 15 features – Mutual information")
plt.xlabel("Mutual information score")
plt.tight_layout()
plt.show()

## 10. Correlation analysis

Correlation is calculated for numerical columns. It is an exploratory analysis and does not use the held-out test set to fit the predictive pipeline.

In [ ]:
correlation = df[numeric_features + [target]].corr(numeric_only=True)
display(correlation.round(2))

plt.figure(figsize=(9, 6))
plt.imshow(correlation, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=75, ha="right")
plt.yticks(range(len(correlation.index)), correlation.index)
plt.title("Numerical feature correlation matrix")
plt.tight_layout()
plt.show()

## 11. Predict for a new applicant

The same fitted pipeline automatically applies the training-time imputers, scaling, and encoding.

In [ ]:
new_applicant = pd.DataFrame([{
    "age":  thirty if False else 30,
    "annual_income": 52000,
    "loan_amount": 15000,
    "credit_score": 710,
    "employment_years": 5,
    "debt_to_income_ratio": 0.28,
    "education": "Graduate",
    "employment_type": "Salaried",
    "home_ownership": "Rent",
    "loan_purpose": "Education"
}])

prediction = model_pipeline.predict(new_applicant)[0]
probability = model_pipeline.predict_proba(new_applicant)[0, 1]
print("Predicted loan_default:", int(prediction))
print("Estimated default probability:", round(float(probability), 3))

## 12. Conclusion

A reusable Scikit-learn preprocessing and modeling pipeline was implemented. Numerical and categorical data are processed separately, missing values are handled, numerical features are scaled, and categorical features are one-hot encoded. Splitting the dataset before fitting transformations and keeping preprocessing inside the pipeline helps prevent data leakage. Feature rankings were examined using Random Forest importance and mutual information.

**Responsible-use note:** This sample is educational only. A real credit decision system requires representative, legally obtained data, fairness and privacy checks, domain validation, human oversight, and a safe fallback process. Model predictions should not be used as the sole basis for approving or rejecting real applicants.